# Inference pipeline

At this point, you should have the following files for the models trained on data from BWC batches 160 to 200:

1. `models/stage1_best_model.joblib`
2. `models/stage2_best_model.joblib`

You should also have unseen gradesheet data (in .xlsx) that you want to run inference on in `data/unseen` folder.

To run evaluation against known ground truth labels, ground truth data (in .xlsx) must be in `data/labels` folder. 

This notebook is currently running inference on data from BWC batches 201 and onwards. Predictions will be saved to `predictions` folder.

## Imports, constants

In [1]:
import pandas as pd
import joblib
from pathlib import Path

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

MODULE_1 = [
    "GH 1",
    "GH 2",
    "GH 3",
    "GH 4",
    "GH 5",
    "GH 6",
    "GH 7",
    "GH 8",
    "GH 9",
    "GH 10",
    "GH 11",
    "GH 12",
    "GH 13",
    "GH 14",
    "GH 15",
    "GH 17",
    "GH 19",
    "GH 21",
    "GH 22",
]

MODULE_2 = [
    "GH 24",
    "GH 25",
    "GH 27",
    "GH 28",
    "GH 29",
    "GH 30",
    "GH 31",
    "GH 32",
    "GH 33",
    "GH 34",
    "GHT",
]

MODULE_3 = ["IF 1", "IF 2", "IF 3", "IF 4", "IF 5", "IF 6", "IFT"]

MODULE_6 = ["AN 1", "AN 2", "AN 3", "AN 4", "AN 5", "AN 6", "AN 7"]

LESSONS_TO_KEEP = MODULE_1 + MODULE_2 + MODULE_3

## Prepare data for inference

### Read raw unseen data

In [2]:
# Read raw unseen data
dfs = []
for file in Path("data/unseen").glob("*.xlsx"):
    df = pd.read_excel(file)
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 143163 entries, 0 to 143162
Data columns (total 73 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   STUDENT_ID                      143163 non-null  str           
 1   STUDENT_FIRST_NAME              143163 non-null  str           
 2   STUDENT_LAST_NAME               143163 non-null  str           
 3   STUDENT_DISPLAY_NAME            143163 non-null  str           
 4   STUDENT_TITLE                   143163 non-null  str           
 5   STUDENT_RANK                    0 non-null       float64       
 6   STUDENT_RANK_ABBREV             0 non-null       float64       
 7   COURSE                          86974 non-null   str           
 8   CLASS_TEMP_DS_ID                86974 non-null   str           
 9   CLASS                           143163 non-null  str           
 10  CLASS_DS_ID                     86974 non-null   str           
 11

### Clean data

In [3]:
# Drop duplicate rows
df = df.drop_duplicates()

# Drop rows with null values in key columns
df = df.dropna(subset=["STUDENT_ID", "LESSON_NUMBER", "EMP_EVAL_SCORE"])

# Keep only the relevant lesson numbers (modules 1 to 3)
# This step also removes rows with suffixes.
df = df[df["LESSON_NUMBER"].isin(LESSONS_TO_KEEP)]

# Drop rows with NG in EMP_EVAL_COMMENTS
df = df[
    ~df["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bNG\b", case=False, na=False)
]

# Drop rows with DNCO
df = df[
    ~(
        df["DUTY_STATUS_DESCRIPTION"].astype("string").eq("DNCO")
        | df["EMP_EVAL_COMMENTS"]
        .astype("string")
        .str.contains(r"\bDNCO\b", case=False, na=False)
    )
]

# Keep only the first row for a given STUDENT_ID and LESSON_NUMBER
df = df.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER", "LESSON_ACTUAL_START_DATE"])

# Reorder columns for readability
first_cols = [
    "STUDENT_ID",
    "LESSON_NUMBER",
    "LESSON_ACTUAL_START_DATE",
    "LESSON_ACTUAL_END_DATE",
    "EMP_EVAL_SCORE",
    "EMP_EVAL_COMMENTS",
    "DUTY_STATUS_DESCRIPTION",
    "ADJUSTED_SCORE",
    "AVERAGE_SCORE",
    "OBJECTIVE_DESCRIPTION",
    "OBJECTIVE_RAW_SCORE",
    "OBJECTIVE_SCORE_WEIGHT",
    "OBJECTIVE_WEIGHTED_SCORE",
    "OBJECTIVE_CRITICAL",
    "OBJECTIVE_ACCEPTABLE_SCORE",
    "OBJECTIVE_ACCEPTABLE_SCORE_DSC",
]
remaining_cols = [col for col in df.columns if col not in first_cols]
df = df[first_cols + remaining_cols]

# Sort by STUDENT_ID
df = df.sort_values(by=["STUDENT_ID", "LESSON_ACTUAL_START_DATE"]).reset_index(drop=True)

### Pivot

In [4]:
# Pivot
wide = (
    df.pivot_table(
        index="STUDENT_ID",
        columns="LESSON_NUMBER",
        values="EMP_EVAL_SCORE",
        aggfunc="first",
    )
    .reset_index()
)

# Reorder columns
wide = wide[["STUDENT_ID"] + LESSONS_TO_KEEP]
wide.columns.name = None
wide

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GH 33,GH 34,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT
0,201EEZ,5.179442,5.267281,5.234087,4.355000,4.045916,4.367075,4.779347,3.446281,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201HLEE,4.624989,4.917051,4.332565,4.444498,4.816778,3.785408,4.155844,4.118668,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,NaN,4.101320,4.173327,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,NaN,4.357262,4.421186,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591
4,201KOHJ,5.113508,4.917051,4.266631,4.425050,4.351428,4.299694,3.487584,3.593985,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,207LUH,5.420786,4.794602,4.912946,4.866944,4.574162,4.505554,4.466744,4.434889,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95,207SEAHI,5.113508,5.342156,4.528630,5.110352,5.084466,4.620851,4.466744,4.513689,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,207SOHY,4.871720,4.977205,4.657893,4.957826,4.290271,4.736512,4.926931,4.198177,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,207TANK,4.994169,5.210692,4.266631,4.910062,4.727366,4.335183,4.428571,4.173113,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Run inference

In [5]:
# Load trained model
stage1_best_model = joblib.load("models/stage1_best_model.joblib")
stage2_best_model = joblib.load("models/stage2_best_model.joblib")

### Stage 1: Predict BWC pass or fail

In [6]:
# Keep only the feature columns
X1 = wide.drop(columns="STUDENT_ID")

# Get predictions and associated probability
y_pred1 = stage1_best_model.predict(X1)
y_score1 = stage1_best_model.predict_proba(X1)[:, 1]

# Append back to wide span table
wide["pred_bwc"] = y_pred1
wide["prob_bwc"] = y_score1
assert wide["pred_bwc"].isnull().sum() == 0
wide

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,pred_bwc,prob_bwc
0,201EEZ,5.179442,5.267281,5.234087,4.355000,4.045916,4.367075,4.779347,3.446281,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.118992
1,201HLEE,4.624989,4.917051,4.332565,4.444498,4.816778,3.785408,4.155844,4.118668,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.065594
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,4.173327,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287,1.0,0.881256
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,4.421186,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591,1.0,0.902146
4,201KOHJ,5.113508,4.917051,4.266631,4.425050,4.351428,4.299694,3.487584,3.593985,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.014283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,207LUH,5.420786,4.794602,4.912946,4.866944,4.574162,4.505554,4.466744,4.434889,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.078722
95,207SEAHI,5.113508,5.342156,4.528630,5.110352,5.084466,4.620851,4.466744,4.513689,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.026853
96,207SOHY,4.871720,4.977205,4.657893,4.957826,4.290271,4.736512,4.926931,4.198177,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.050677
97,207TANK,4.994169,5.210692,4.266631,4.910062,4.727366,4.335183,4.428571,4.173113,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.027626


### Stage 2: Among predicted BWC passes, predict fighter or not

In [7]:
# Keep only predicted passes
passes = wide[wide["pred_bwc"] == 1].drop(columns=["pred_bwc", "prob_bwc"])

# Keep only the feature columns
X2 = passes.drop(columns=["STUDENT_ID"])

# Get predictions and associated probability
y_pred2 = stage2_best_model.predict(X2)
y_score2 = stage2_best_model.predict_proba(X2)[:, 1]

# Append back to wide span table
passes["pred_fighter"] = y_pred2
passes["prob_fighter"] = y_score2
assert passes["pred_fighter"].isnull().sum() == 0
passes

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GHT,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,pred_fighter,prob_fighter
2,201HONT,5.113508,4.917051,4.133894,4.671514,4.773037,4.735986,4.372067,3.616723,NaN,...,4.173327,5.027020,4.245510,4.539404,4.406282,3.777039,4.195178,4.557287,0.0,0.365022
3,201KHOOC,5.301836,4.977205,4.503209,4.634096,4.935050,4.214286,4.815559,4.484804,NaN,...,4.421186,5.377912,5.277256,4.344802,4.386612,NaN,4.888308,4.204591,1.0,0.523451
5,201KOHZ,5.813411,6.180373,5.756446,4.930210,4.751955,4.814494,5.047512,5.502013,NaN,...,3.965473,5.071061,4.491836,4.585104,3.962011,4.507744,3.785039,4.760613,1.0,0.699344
6,201KSARAN,5.301836,4.977205,4.398210,4.249666,4.473469,4.039001,4.233766,4.197280,NaN,...,3.926079,4.270524,4.368571,4.306142,4.388513,3.631730,3.957875,4.258857,0.0,0.234078
7,201LEEJ,5.116952,4.977205,5.101195,5.016398,5.315733,4.388821,4.257563,4.376099,NaN,...,4.702097,4.628361,4.860992,4.438608,4.468097,4.295469,3.913083,4.211606,0.0,0.467614
8,201LIMK,6.084082,5.653140,5.051111,4.808638,4.633721,4.890357,4.692850,4.456488,NaN,...,4.789537,4.315432,4.409637,4.278003,3.924180,NaN,NaN,NaN,0.0,0.470441
9,201ONGE,5.116952,4.972374,4.398210,4.813387,4.713961,4.731633,4.488572,4.641768,NaN,...,4.342767,4.225564,3.667171,4.657209,4.244153,4.449425,4.486976,4.578947,1.0,0.536148
10,201PANGM,5.398251,5.267281,4.975985,5.001205,4.792674,5.180113,4.896214,5.325728,NaN,...,4.036151,4.672861,4.601960,4.383548,4.428227,4.260308,4.519801,4.568555,1.0,0.539588
11,201TANGK,4.670175,4.574422,4.173120,4.784895,5.006211,4.334886,4.135399,4.117753,NaN,...,4.286667,4.583810,5.163993,4.422381,4.468097,3.481036,4.163149,4.385917,0.0,0.440448
12,202ANGK,5.286686,4.908005,4.430533,4.516597,5.039315,4.489297,4.477631,4.909192,NaN,...,4.226671,4.806054,4.656508,4.422381,4.519171,4.054513,4.118079,4.740115,0.0,0.459371


### Final output

In [8]:
predictions = pd.merge(
    wide[["STUDENT_ID", "pred_bwc", "prob_bwc"]],
    passes[["STUDENT_ID", "pred_fighter", "prob_fighter"]],
    on="STUDENT_ID",
    how="left",
)
predictions = predictions.fillna(0.0)

In [9]:
predictions.head(20)

,STUDENT_ID,pred_bwc,prob_bwc,pred_fighter,prob_fighter
0,201EEZ,0.0,0.118992,0.0,0.000000
1,201HLEE,0.0,0.065594,0.0,0.000000
2,201HONT,1.0,0.881256,0.0,0.365022
3,201KHOOC,1.0,0.902146,1.0,0.523451
4,201KOHJ,0.0,0.014283,0.0,0.000000
5,201KOHZ,1.0,0.741961,1.0,0.699344
6,201KSARAN,1.0,0.938351,0.0,0.234078
7,201LEEJ,1.0,0.939199,0.0,0.467614
8,201LIMK,1.0,0.752536,0.0,0.470441
9,201ONGE,1.0,0.933237,1.0,0.536148


In [10]:
predictions.tail(20)

,STUDENT_ID,pred_bwc,prob_bwc,pred_fighter,prob_fighter
79,206TANA,0.0,0.034931,0.0,0.000000
80,206TANN,0.0,0.195341,0.0,0.000000
81,206TANS,0.0,0.139525,0.0,0.000000
82,206YEOG,0.0,0.458989,0.0,0.000000
83,206YEWP,1.0,0.525294,1.0,0.713748
84,207ANGD,0.0,0.027212,0.0,0.000000
85,207HAZWAN,0.0,0.094309,0.0,0.000000
86,207JLIEW,0.0,0.208354,0.0,0.000000
87,207KANGB,0.0,0.046364,0.0,0.000000
88,207KOHQ,0.0,0.112053,0.0,0.000000


In [11]:
predictions.to_csv("predictions/201-207_predictions.csv", index=False)

## Evaluate against known ground truth

Ground truth is taken from `data/labels/BWC201t207_Status_as_of_end_Feb2026.xlsx`, as sent by Tommy to Melody on Defence mail on 13 March 2026.

### Get ground truth

In [12]:
# Load ground truth
ground_truth = pd.read_excel("data/labels/BWC201t207_Status_as_of_end_Feb2026.xlsx")
ground_truth = ground_truth.rename(columns={"STUDENT_ID_ORIG": "STUDENT_ID"})

# Create target columns
ground_truth["target_bwc"] = pd.NA
ground_truth.loc[ground_truth["BWC_Status"] == "Fail", "target_bwc"] = 0.0
ground_truth.loc[
    ground_truth["BWC_Status"].isin(["Pass", "DNF(SUPT)"]), "target_bwc"
] = 1.0
ground_truth["target_bwc"].astype("Float64")

ground_truth["target_fighter"] = pd.NA
ground_truth.loc[
    ((ground_truth["target_bwc"].notna()) & (ground_truth["Stream_Group"] != "Fighter")),
    "target_fighter",
] = 0.0
ground_truth.loc[
    ((ground_truth["target_bwc"] == 1.0) & (ground_truth["Stream_Group"] == "Fighter")),
    "target_fighter",
] = 1.0
ground_truth["target_bwc"].astype("Float64")

if len(ground_truth) != len(predictions):
    print(set(ground_truth["STUDENT_ID"].unique()) - set(predictions["STUDENT_ID"].unique()))
    print(set(predictions["STUDENT_ID"].unique()) - set(ground_truth["STUDENT_ID"].unique()))

{'202LIMN'}
set()


In [13]:
final = pd.merge(ground_truth, predictions, on="STUDENT_ID", how="left")

# Drop rows where target_BWC is nan, i.e. Ongoing, DNF due to med, disciplinary, etc.
final = final.dropna(subset="target_bwc").reset_index(drop=True)

# Enforce datatype
for col in ["target_bwc", "target_fighter", "pred_bwc", "pred_fighter", "prob_bwc", "prob_fighter"]:
    final[col] = final[col].astype("Float64")
    
final

,STUDENT_ID,Batch,BWC_Status,Stream_Group,Stream_Unit,target_bwc,target_fighter,pred_bwc,prob_bwc,pred_fighter,prob_fighter
0,201EEZ,201,Fail,NaN,NaN,0.0,0.0,0.0,0.118992,0.0,0.0
1,201HLEE,201,Fail,NaN,NaN,0.0,0.0,0.0,0.065594,0.0,0.0
2,201HONT,201,Fail,NaN,NaN,0.0,0.0,1.0,0.881256,0.0,0.365022
3,201KOHJ,201,Fail,NaN,NaN,0.0,0.0,0.0,0.014283,0.0,0.0
4,201LIMK,201,Fail,NaN,NaN,0.0,0.0,1.0,0.752536,0.0,0.470441
5,201KHOOC,201,Pass,Heli,RWC,1.0,0.0,1.0,0.902146,1.0,0.523451
6,201KOHZ,201,Pass,Fighter,FWC,1.0,1.0,1.0,0.741961,1.0,0.699344
7,201KSARAN,201,Pass,Transport,TWC,1.0,0.0,1.0,0.938351,0.0,0.234078
8,201LEEJ,201,Pass,Fighter,FWC,1.0,1.0,1.0,0.939199,0.0,0.467614
9,201ONGE,201,Pass,Fighter,FWC,1.0,1.0,1.0,0.933237,1.0,0.536148


### Evaluation

In [14]:
final["pred_fighter"].value_counts(dropna=False)

pred_fighter
0.0    39
1.0    21
Name: count, dtype: Int64

In [15]:
print("Stage 1: Predict pass or fail BWC")
print(
    classification_report(
        final["target_bwc"].astype("Int64"),
        final["pred_bwc"].astype("Int64"),
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(final['target_bwc'], final['prob_bwc']):.4f}")
tn, fp, fn, tp = confusion_matrix(final["target_bwc"], final["pred_bwc"]).ravel()
print("Confusion Matrix:")
print(f"TN: {tn}, FP: {fp}")
print(f"FN: {fn}, TP: {tp}")

print()

print("Stage 2: Among predicted passes, predict streamed to fighter or not")
print(
    classification_report(
        final["target_fighter"].astype("Int64"),
        final["pred_fighter"].astype("Int64"),
        zero_division=0,
        digits=4,
    )
)
print(f"ROC-AUC: {roc_auc_score(final['target_fighter'], final['prob_fighter']):.4f}")
tn, fp, fn, tp = confusion_matrix(
    final["target_fighter"], final["pred_fighter"]
).ravel()
print("Confusion Matrix:")
print(f"TN: {tn}, FP: {fp}")
print(f"FN: {fn}, TP: {tp}")

Stage 1: Predict pass or fail BWC
              precision    recall  f1-score   support

         0.0     0.9412    0.8889    0.9143        18
         1.0     0.9535    0.9762    0.9647        42

    accuracy                         0.9500        60
   macro avg     0.9473    0.9325    0.9395        60
weighted avg     0.9498    0.9500    0.9496        60

ROC-AUC: 0.9577
Confusion Matrix:
TN: 16, FP: 2
FN: 1, TP: 41

Stage 2: Among predicted passes, predict streamed to fighter or not
              precision    recall  f1-score   support

         0.0     0.8205    0.8889    0.8533        36
         1.0     0.8095    0.7083    0.7556        24

    accuracy                         0.8167        60
   macro avg     0.8150    0.7986    0.8044        60
weighted avg     0.8161    0.8167    0.8142        60

ROC-AUC: 0.8958
Confusion Matrix:
TN: 32, FP: 4
FN: 7, TP: 17


In [16]:
final.to_csv("predictions/201-206_predictions_with_ground_truth.csv", index=False)